In [15]:
import pandas as pd
import numpy as np
import os
import joblib
import math

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from gensim.models import Word2Vec

In [16]:
import pandas as pd

df = pd.read_csv("../data/cleaned_imdb_dataset.csv")

texts = df["cleaned_review"].astype(str).tolist()

tokenized_reviews = [review.split() for review in texts]

print("Dataset Shape:", df.shape)
print("Total Reviews:", len(texts))
print("Tokenized Reviews:", len(tokenized_reviews))

Dataset Shape: (50000, 2)
Total Reviews: 50000
Tokenized Reviews: 50000


In [17]:
tfidf_vectorizer = TfidfVectorizer()

tfidf_matrix = tfidf_vectorizer.fit_transform(texts)

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

total_elements = tfidf_matrix.shape[0] * tfidf_matrix.shape[1]

zero_elements = total_elements - tfidf_matrix.nnz

sparsity = (zero_elements / total_elements) * 100

print("Sparsity:", sparsity, "%")

print("Vocabulary Size:", len(tfidf_vectorizer.vocabulary_))

TF-IDF Matrix Shape: (50000, 93350)
Sparsity: 99.89611648634173 %
Vocabulary Size: 93350


In [18]:
feature_names = tfidf_vectorizer.get_feature_names_out()

idf_values = tfidf_vectorizer.idf_

idf_table = pd.DataFrame({
    "Word": feature_names,
    "IDF Score": idf_values
})

print("Words with HIGH IDF:")
print(idf_table.sort_values(by="IDF Score", ascending=False).head(10))

print("\nWords with LOW IDF:")
print(idf_table.sort_values(by="IDF Score", ascending=True).head(10))

Words with HIGH IDF:
              Word  IDF Score
93349        כרמון  11.126651
93348         יגאל  11.126651
93347    żmijewski  11.126651
2      00000000000  11.126651
93331       önsjön  11.126651
93329          ïts  11.126651
93328           ïn  11.126651
93327          île  11.126651
93326         ísnt  11.126651
93324       êxtase  11.126651

Words with LOW IDF:
        Word  IDF Score
55534  movie   1.492286
58135    not   1.518172
30245   film   1.519113
59321    one   1.564422
50660   make   1.659462
48448   like   1.707794
72984    see   1.711902
33670    get   1.863007
83080   time   1.892594
34688   good   1.967551


In [19]:
first_review_vector = tfidf_matrix[0].toarray()[0]

word_weights = pd.DataFrame({
    "Word": feature_names,
    "TF-IDF Weight": first_review_vector
})

word_weights = word_weights[word_weights["TF-IDF Weight"] > 0]

word_weights = word_weights.sort_values(
    by="TF-IDF Weight",
    ascending=False
)

print("Highest TF-IDF words in the first review:")
print(word_weights.head(10))

print("\nLowest non-zero TF-IDF words in the first review:")
print(word_weights.tail(10))

Highest TF-IDF words in the first review:
              Word  TF-IDF Weight
60508           oz       0.465830
88854     violence       0.204669
65087       prison       0.188239
41723      inmates       0.174131
48900           ll       0.167705
31429       forget       0.140465
42500      inwards       0.125289
61953  penitentary       0.125289
32933     gangstas       0.119319
78949       strike       0.116103

Lowest non-zero TF-IDF words in the first review:
         Word  TF-IDF Weight
87605     use       0.033526
72101  scenes       0.033268
28078    ever       0.030880
57245   never       0.030489
51196    many       0.030330
34120    give       0.027356
90228    well       0.025656
91762   would       0.024872
57769      no       0.024836
59321     one       0.018282


In [20]:
toy_documents = [
    "good movie",
    "good good acting",
    "bad movie",
    "good acting"
]

In [21]:
N = len(toy_documents)

word = "good"

df_word = sum(word in doc.split() for doc in toy_documents)

tf = toy_documents[0].split().count(word)

idf = math.log((N + 1) / (df_word + 1)) + 1

manual_tfidf = tf * idf

print("Total Documents:", N)
print("Document Frequency of good:", df_word)
print("Term Frequency:", tf)
print("IDF:", idf)
print("Manual TF-IDF:", manual_tfidf)

Total Documents: 4
Document Frequency of good: 3
Term Frequency: 1
IDF: 1.2231435513142097
Manual TF-IDF: 1.2231435513142097


In [22]:
toy_vectorizer = TfidfVectorizer(norm=None)

toy_matrix = toy_vectorizer.fit_transform(toy_documents)

feature_names = toy_vectorizer.get_feature_names_out()

good_index = list(feature_names).index("good")

library_tfidf = toy_matrix[0, good_index]

print("Manual TF-IDF:", manual_tfidf)
print("Library TF-IDF:", library_tfidf)

Manual TF-IDF: 1.2231435513142097
Library TF-IDF: 1.2231435513142097


In [23]:
first_document = toy_documents[0]

words = first_document.split()

manual_bigram = words[0] + " " + words[1]

print("Manual Bigram:", manual_bigram)

Manual Bigram: good movie


In [24]:
bigram_vectorizer = CountVectorizer(ngram_range=(2, 2))

bigram_matrix = bigram_vectorizer.fit_transform(toy_documents)

print("Library Bigrams:")
print(bigram_vectorizer.get_feature_names_out())

Library Bigrams:
['bad movie' 'good acting' 'good good' 'good movie']


In [ ]:
skipgram_model = Word2Vec(
    sentences=tokenized_reviews,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=10,
    seed=42
)

print("Word2Vec Skip-gram Model Trained Successfully!")

import os

os.makedirs("../models", exist_ok=True)

skipgram_model.save("../models/skipgram_model.model")

print("Skip-gram model saved successfully!")

Word2Vec Skip-gram Model Trained Successfully!


In [26]:
print("Vector Size:", skipgram_model.vector_size)
print("Window Size:", skipgram_model.window)
print("Minimum Count:", skipgram_model.min_count)
print("Epochs:", skipgram_model.epochs)
print("Vocabulary Size:", len(skipgram_model.wv))

Vector Size: 100
Window Size: 5
Minimum Count: 2
Epochs: 10
Vocabulary Size: 66887


In [27]:
query_words = ["movie", "film", "acting", "story", "good"]

for word in query_words:

    print("\nSimilar words for:", word)

    if word in skipgram_model.wv:

        print(skipgram_model.wv.most_similar(word, topn=5))

    else:

        print("Word not found in vocabulary.")


Similar words for: movie
[('film', 0.8707020282745361), ("havn't", 0.7993521690368652), ('film!the', 0.7885066866874695), ('strangeland', 0.7815086841583252), ('atleast', 0.7809796929359436)]

Similar words for: film
[('movie', 0.8707022070884705), ("'cut'", 0.8017076849937439), ('towelhead', 0.7887894511222839), ('gorefests', 0.7753452658653259), ('indications', 0.7679910659790039)]

Similar words for: acting
Word not found in vocabulary.

Similar words for: story
[('storyline', 0.8186007142066956), ('plot', 0.7660120725631714), ('tale', 0.7645995616912842), ('stories', 0.763988733291626), ('snorer', 0.7035160064697266)]

Similar words for: good
[('decent', 0.8108570575714111), ('great', 0.7956819534301758), ('film!the', 0.7613853216171265), ('bad', 0.7594527006149292), ('nice', 0.7486985325813293)]


In [30]:
from gensim.models import Word2Vec

cbow_model = Word2Vec.load("../models/cbow_model.model")

print("CBOW model loaded successfully!")

skipgram_model = Word2Vec.load("../models/skipgram_model.model")
print("Skip-gram model loaded successfully!")

fasttext_model = Word2Vec.load("../models/fasttext_model.model")
print("FastText model loaded successfully!")

models = {
    "CBOW": cbow_model,
    "Skip-gram": skipgram_model,
    "FastText": fasttext_model
}

for model_name, model in models.items():

    print("\n==========", model_name, "==========")

    for word in query_words:

        print("\nSimilar words for:", word)

        if word in model.wv:

            print(model.wv.most_similar(word, topn=5))

        else:

            print("Word not found.")

CBOW model loaded successfully!


FileNotFoundError: [Errno 2] No such file or directory: '../models/skipgram_model.model'

In [ ]:
word_pairs = [
    ("movie", "film"),
    ("movie", "banana")
]

for model_name, model in models.items():

    print("\n==========", model_name, "==========")

    for word1, word2 in word_pairs:

        if word1 in model.wv and word2 in model.wv:

            similarity = model.wv.similarity(word1, word2)

            print(word1, "and", word2, "=", similarity)

        else:

            print("One of the words is not available.")

In [ ]:
try:

    result = skipgram_model.wv.most_similar(
        positive=["good", "terrible"],
        negative=["bad"],
        topn=5
    )

    print("Analogy Results:")
    print(result)

except KeyError as e:

    print("Word not found:", e)

In [ ]:
os.makedirs("../models", exist_ok=True)

joblib.dump(
    tfidf_vectorizer,
    "../models/tfidf_vectorizer.pkl"
)

skipgram_model.save("../models/skipgram.model")

cbow_model.save("../models/cbow.model")

fasttext_model.save("../models/fasttext.model")

print("All models and vectorizer saved successfully!")

In [ ]:
import re
import unicodedata

import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

stop_words = set(stopwords.words("english"))

negation_words = {
    "no", "not", "nor", "don't", "didn't",
    "doesn't", "isn't", "wasn't", "weren't",
    "won't", "wouldn't", "couldn't", "shouldn't",
    "can't", "cannot"
}

stop_words = stop_words - negation_words

lemmatizer = WordNetLemmatizer()


def clean_text(text):

    text = unicodedata.normalize("NFKC", str(text))

    text = text.lower()

    text = re.sub(r"<[^>]+>", "", text)

    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    text = re.sub(r"[^\w\s'!?]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    words = text.split()

    words = [word for word in words if word not in stop_words]

    words = [
        lemmatizer.lemmatize(word, wordnet.VERB)
        for word in words
    ]

    return " ".join(words)

In [ ]:
def vectorize_document(raw_document, representation="tfidf"):

    cleaned_document = clean_text(raw_document)

    if representation == "tfidf":

        return tfidf_vectorizer.transform([cleaned_document])

    elif representation == "skipgram":

        words = cleaned_document.split()

        vectors = [
            skipgram_model.wv[word]
            for word in words
            if word in skipgram_model.wv
        ]

        if len(vectors) == 0:

            return np.zeros(skipgram_model.vector_size)

        return np.mean(vectors, axis=0)

    else:

        raise ValueError("Choose 'tfidf' or 'skipgram'")

In [ ]:
sample_review = "This movie was amazing and the acting was excellent!"

tfidf_result = vectorize_document(
    sample_review,
    "tfidf"
)

skipgram_result = vectorize_document(
    sample_review,
    "skipgram"
)

print("TF-IDF Vector Shape:", tfidf_result.shape)

print("Skip-gram Vector Shape:", skipgram_result.shape)

In [ ]:
print("MEMBER 2 WORK COMPLETED")

print("\nTF-IDF Matrix:", tfidf_matrix.shape)

print("Skip-gram Vocabulary:", len(skipgram_model.wv))

print("CBOW Vocabulary:", len(cbow_model.wv))

print("FastText Vocabulary:", len(fasttext_model.wv))

print("\nSaved Files:")
print(os.listdir("../models"))